# AI for Handwriting Identification — Colab quickstart

**Colab has no `C:\Users\...`.** On your PC the release may live at `C:\Users\thisb\Downloads\AnyScriptFiltered.tar.gz`. For Colab: **upload that `.tar.gz` to Google Drive** (e.g. `MyDrive/`), then use a **`/content/drive/MyDrive/...`** path below and extract there so training survives disconnects.

**Dataset:** `DATA_ROOT` = folder whose **children are author IDs** (default `/content/drive/MyDrive/AnyScriptFiltered`). Layouts: `author/book/*.jpg` or `author/*.png`.

## Part A, B, C (scripts are in `scripts/` after `%cd` there)

| Part | Script | Output |
|------|--------|--------|
| **A — Train** | `train_triplet_unsloth.py` | `OUT/best.pt` |
| **B — FAISS** | `build_faiss_index.py` | `OUT/faiss.index`, `OUT/meta.npy` |
| **C — Submit CSV** | `export_anyscript_submission.py` | dense `intra_book` / `extra_book` CSV |

More detail: **`README.md`** (Quick start + ICDAR submission).

In [ ]:
# Optional: persist data & checkpoints on Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone repo (fork/URL ok — change if you renamed the repo)
!rm -rf /content/ai-hw
!git clone https://github.com/Todd838/AI-for-Handwriting-Identification.git /content/ai-hw
%cd /content/ai-hw/scripts

In [ ]:
# Install everything from the repo (same as local requirements.txt)
%pip install -q -r /content/ai-hw/requirements.txt
# If pip errors on a package, run: %pip install -q unsloth  (after GPU runtime is on)

In [ ]:
# Unsloth (use Runtime → GPU first). If install fails, train with --no_unsloth
import torch
print("CUDA available:", torch.cuda.is_available())
%pip install -q unsloth

In [ ]:
# Dataset on Colab = Drive path, NOT C:\Users\thisb\Downloads\...
# Upload AnyScriptFiltered.tar.gz from your PC to Drive, then set ARCHIVE to that file.
import os
import subprocess

ARCHIVE = "/content/drive/MyDrive/AnyScriptFiltered.tar.gz"
EXTRACT_PARENT = "/content/drive/MyDrive"
DATA_ROOT = "/content/drive/MyDrive/AnyScriptFiltered"  # adjust if your tree lives elsewhere

if os.path.isdir(DATA_ROOT):
    print("OK: DATA_ROOT ->", DATA_ROOT)
elif os.path.isfile(ARCHIVE):
    print("Extracting (can take a long time; needs Drive space)...")
    os.makedirs(EXTRACT_PARENT, exist_ok=True)
    subprocess.run(["tar", "-xzf", ARCHIVE, "-C", EXTRACT_PARENT], check=False)
    print("If DATA_ROOT is wrong, find author/book folders and set DATA_ROOT to that path.")
    print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT))
else:
    print("Upload AnyScriptFiltered.tar.gz to Drive, or fix ARCHIVE path.")

## Part A — training

Uncomment the `!python` line when `DATA_ROOT` exists. Checkpoints go to **`OUT`** on Drive.

In [ ]:
OUT = "/content/drive/MyDrive/anyscript_runs/run1"
# !python train_triplet_unsloth.py --data_root {DATA_ROOT} --model_name THUDM/glm-ocr --output_dir {OUT} --epochs 1 --steps_per_epoch 200 --batch_size 2
# Add --no_unsloth if Unsloth fails

## Part B — FAISS index

Run after **`best.pt`** exists under `OUT`. Uncomment the next cell.

In [ ]:
# Part B streams images from disk in batches (safe for huge galleries). Lower --batch_size if GPU OOM.
# %cd /content/ai-hw/scripts
# !python build_faiss_index.py --data_root {DATA_ROOT} --checkpoint {OUT}/best.pt --index_out {OUT}/faiss.index --meta_out {OUT}/meta.npy --batch_size 4
# Add --no_unsloth if needed (match Part A).

## Part C — submission CSV

Set **`QUERY_ROOT`** to held-out query images. **`GALLERY_ROOT`** is usually **`DATA_ROOT`**. For a real upload, add **`--query_ids_json`** and **`--gallery_ids_json`** (see **`README.md`**). **`--allow_synthetic_ids`** is only for testing.

In [ ]:
# QUERY_ROOT = "/content/drive/MyDrive/anyscript_query_holdout"
# GALLERY_ROOT = DATA_ROOT
# %cd /content/ai-hw/scripts
# !python export_anyscript_submission.py --mode intra_book --checkpoint {OUT}/best.pt --gallery_data_root {GALLERY_ROOT} --query_data_root {QUERY_ROOT} --out_csv {OUT}/submission_intra_book.csv --batch_size 4 --allow_synthetic_ids
# !python export_anyscript_submission.py --mode extra_book --checkpoint {OUT}/best.pt --gallery_data_root {GALLERY_ROOT} --query_data_root {QUERY_ROOT} --out_csv {OUT}/submission_extra_book.csv --batch_size 4 --allow_synthetic_ids